In [ ]:
from IPython.display import display, Image
display(Image('loss_curves.png'))
display(Image('benchmark_comparison.png'))
display(Image('sample_forecasts.png'))


In [7]:
import os
if os.path.exists("./chronos-pse-finetuned/config.json"):
    print("Training Complete")
else:
    print("Training still in progress or failed")
    # Check for logs
    if os.path.exists("./chronos-pse-results"):
        print("Last checkpoint logs:")
        !ls ./chronos-pse-results


Training Complete


In [6]:
import tarfile
import os

def export_model(model_dir, output_filename):
    with tarfile.open(output_filename, "w:gz") as tar:
        tar.add(model_dir, arcname=os.path.basename(model_dir))
    print(f"Model exported to {output_filename}")

export_model("./chronos-pse-finetuned", "chronos_pse_finetuned.tar.gz")

from google.colab import files
# files.download("chronos_pse_finetuned.tar.gz") # Uncomment to download in browser


Model exported to chronos_pse_finetuned.tar.gz


In [5]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from chronos import Chronos2Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

def calculate_mape(y_true, y_pred):
    y_true = np.where(y_true == 0, 1e-9, y_true)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def benchmark_model(model_path, val_data, prediction_length=24):
    print(f"Loading model from {model_path}...")
    pipeline = Chronos2Pipeline.from_pretrained(model_path, device_map="auto")

    results = []
    for i, item in enumerate(val_data[:10]):
        full_series = item["target"]
        if len(full_series) < prediction_length + 64:
            continue

        history = full_series[:-prediction_length]
        ground_truth = full_series[-prediction_length:]

        df = pd.DataFrame({
            "timestamp": pd.date_range(start="2023-01-01", periods=len(history), freq="D"),
            "target": history,
            "item_id": f"stock_{i}"
        })

        pred_df = pipeline.predict_df(
            df,
            prediction_length=prediction_length,
            quantile_levels=[0.1, 0.5, 0.9],
            id_column="item_id"
        )

        median_pred = pred_df["0.5"].values

        mae = mean_absolute_error(ground_truth, median_pred)
        rmse = np.sqrt(mean_squared_error(ground_truth, median_pred))
        mape = calculate_mape(ground_truth, median_pred)

        results.append({"MAE": mae, "RMSE": rmse, "MAPE": mape})

    return pd.DataFrame(results).mean().to_dict()

# Load Val Data
with open("/content/val.jsonl", "r") as f:
    val_inputs = [json.loads(line) for line in f]

# 1. Benchmark Fine-tuned Model
print("Benchmarking fine-tuned model...")
ft_metrics = benchmark_model("./chronos-pse-finetuned", val_inputs)
print(f"Fine-tuned Metrics: {ft_metrics}")

# 2. Benchmark Base Model (Zero-shot)
print("Benchmarking base model (zero-shot)...")
base_metrics = benchmark_model("amazon/chronos-2", val_inputs)
print(f"Base Metrics: {base_metrics}")

# 3. Save Metrics
with open("benchmarks.json", "w") as f:
    json.dump({"fine_tuned": ft_metrics, "base": base_metrics}, f)

# 4. Generate Graphs
with open("train_history.json", "r") as f:
    history = json.load(f)

train_loss = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, train_loss, label='Train Loss', color='blue', linewidth=2)
plt.title('Training Loss Curve (PSE Stock Fine-tuning)')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('loss_curves.png')
plt.close()

metrics_df = pd.DataFrame([
    {"Model": "Base (Zero-shot)", "Metric": "MAE", "Value": base_metrics['MAE']},
    {"Model": "Fine-tuned", "Metric": "MAE", "Value": ft_metrics['MAE']},
    {"Model": "Base (Zero-shot)", "Metric": "RMSE", "Value": base_metrics['RMSE']},
    {"Model": "Fine-tuned", "Metric": "RMSE", "Value": ft_metrics['RMSE']}
])

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x='Metric', y='Value', hue='Model')
plt.title('Benchmark Results: Error Metrics (Lower is Better)')
plt.savefig('benchmark_comparison.png')
plt.close()

pipeline_ft = Chronos2Pipeline.from_pretrained("./chronos-pse-finetuned", device_map="auto")
pipeline_base = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="auto")

fig, axes = plt.subplots(3, 1, figsize=(15, 18))

for i in range(3):
    item = val_inputs[i]
    full_series = item["target"]
    history = full_series[:-24]
    ground_truth = full_series[-24:]

    df = pd.DataFrame({
        "timestamp": pd.date_range(start="2023-01-01", periods=len(history), freq="D"),
        "target": history,
        "item_id": f"sample_{i}"
    })

    pred_ft = pipeline_ft.predict_df(df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9], id_column="item_id")
    pred_base = pipeline_base.predict_df(df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9], id_column="item_id")

    ax = axes[i]
    hist_to_plot = history[-100:]
    hist_x = range(len(hist_to_plot))
    future_x = range(len(hist_to_plot), len(hist_to_plot) + 24)

    ax.plot(hist_x, hist_to_plot, label="History", color="black", alpha=0.6)
    ax.plot(future_x, ground_truth, label="True", color="green", linewidth=2)
    ax.plot(future_x, pred_ft["0.5"], label="Fine-tuned Forecast", color="purple", linestyle="--")
    ax.plot(future_x, pred_base["0.5"], label="Base Forecast", color="orange", linestyle=":")
    ax.fill_between(future_x, pred_ft["0.1"], pred_ft["0.9"], color="purple", alpha=0.1)

    ax.set_title(f"Stock Sample {i+1}: Fine-tuned vs Base")
    ax.legend()
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('sample_forecasts.png')
plt.close()
print("All benchmarks and visualizations completed.")


Benchmarking fine-tuned model...
Loading model from ./chronos-pse-finetuned...
Fine-tuned Metrics: {'MAE': 0.2638481449190294, 'RMSE': 0.3188956528333251, 'MAPE': 4.947278781799334}
Benchmarking base model (zero-shot)...
Loading model from amazon/chronos-2...
Base Metrics: {'MAE': 0.35998496001120656, 'RMSE': 0.40793526263660834, 'MAPE': 6.3518856699570145}
All benchmarks and visualizations completed.


In [3]:
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import TrainingArguments, EarlyStoppingCallback
from chronos import Chronos2Pipeline
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode
from chronos.chronos2.trainer import Chronos2Trainer
import os

# 1. Load Data
def load_jsonl(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            item = json.loads(line)
            data.append({"target": np.array(item["target"], dtype=np.float32)})
    return data

train_inputs = load_jsonl("/content/train.jsonl")
val_inputs = load_jsonl("/content/val.jsonl")

# 2. Setup Model
model_id = "amazon/chronos-2"
print(f"Loading {model_id}...")
pipeline = Chronos2Pipeline.from_pretrained(model_id, device_map="auto")
model = pipeline.model
config = pipeline.model.config

# 3. Datasets
prediction_length = 24
context_length = 512
batch_size = 32

train_dataset = Chronos2Dataset(
    inputs=train_inputs,
    context_length=context_length,
    prediction_length=prediction_length,
    batch_size=batch_size,
    output_patch_size=config.chronos_config["output_patch_size"],
    min_past=64,
    mode=DatasetMode.TRAIN,
)

# For validation, we want a finite dataset to get a stable eval_loss
# We'll limit it to 100 samples (batches)
val_dataset = Chronos2Dataset(
    inputs=val_inputs,
    context_length=context_length,
    prediction_length=prediction_length,
    batch_size=batch_size,
    output_patch_size=config.chronos_config["output_patch_size"],
    min_past=64,
    mode=DatasetMode.VALIDATION,
)

# 4. Training Arguments
# We remove load_best_model_at_end for the first successful run to avoid the KeyError
training_args = TrainingArguments(
    output_dir="./chronos-pse-results",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    max_steps=300,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    dataloader_num_workers=2,
    report_to="none",
    overwrite_output_dir=True,
)

# 5. Trainer
# We'll use a custom evaluation loop if needed, but let's see if this works
trainer = Chronos2Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Starting training...")
trainer.train()

# 6. Save Model
save_path = "./chronos-pse-finetuned"
model.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# 7. Save History
log_history = trainer.state.log_history
with open("train_history.json", "w") as f:
    json.dump(log_history, f)


Loading amazon/chronos-2...


The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training...


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss
50,3.755300,No log
100,3.470800,No log
150,3.274700,No log
200,3.091000,No log
250,3.061600,No log
300,3.066300,No log


Model saved to ./chronos-pse-finetuned


In [4]:
try:
    import chronos
    print("Chronos installed successfully")
except ImportError:
    print("Chronos not found, retrying pip install...")
    !pip install -q chronos-forecasting transformers accelerate


Chronos installed successfully


In [1]:
!find /content -name "*.jsonl"


/content/train.jsonl
/content/val.jsonl


In [3]:
!pip install chronos-forecasting transformers accelerate matplotlib seaborn torch numpy scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 173.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 65.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [9]:
files.download("chronos_pse_finetuned.tar.gz")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>